## Ingest Category Dimension Data into Bronze Layer


import required libraries 

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

In [0]:
# define schema
%run /Workspace/Shopvista_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, bronze_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog","abc","catalog")
dbutils.widgets.text("data_source","efg", "data_souurce")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

In [0]:
# Define schema for the date data

date_schema = StructType([
    StructField("date", DateType(), True),
    StructField("year", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("quarter", IntegerType(), True),

])

In [0]:
#  show the path to the datasource 
base_path = f"s3://shopvista-sv/{data_source}/*csv"

print(base_path)

In [0]:

#load data into dataframe and add metadata
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(date_schema)
        .load(base_path)
        .withColumn("read_timstamp", F.current_timestamp()) # Add metadata columns
        .select("*", "_metadata.file_name", "_metadata.file_path") # Add metadata columns
)



In [0]:
display(df.limit(5))

In [0]:

# write date to bronze table
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")